# 01 - Data Preparation & QA Generation Pipeline

Pipeline stages (each is a script in `src/`, so the notebook is only a runner):

| Stage | Script | Output |
|---|---|---|
| 1. Clean + chunk documents | `data_prep.py` | `data/interim/chunks.jsonl` |
| 2. LLM-assisted QA generation + filters | `build_qa.py` | `data/interim/qa_raw.jsonl` |
| 3. Manual verification of a sample | `review.py` | `data/interim/qa_clean.jsonl` + review report |
| 4. Rule-grounded climate examples (synthetic) | `build_climate.py` | `data/interim/climate.jsonl` |
| 5. Out-of-domain examples + refusals | `build_ood.py` | `data/interim/ood.jsonl` |
| 6. Split by source document + retrieval context | `make_splits.py` | `data/processed/*.jsonl`, `stats.json` |

**Before you start:** put your source documents (PDF/TXT/HTML/MD) in `data/raw/` and list each one in `data/raw/sources.csv` (see `sources_template.csv`). Check every licence. Use a **T4 GPU** runtime for stage 2.

## 0. Setup

In [ ]:
!nvidia-smi -L
!pip install -q pypdf "transformers>=4.46" "accelerate>=1.0" bitsandbytes pandas matplotlib
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json
PROJECT = '/content/drive/MyDrive/crop-advisory-assistant'   # <- change if needed
os.chdir(PROJECT); sys.path.insert(0, f'{PROJECT}/src')
print(os.listdir('data/raw')[:20])

In [ ]:
# Sanity-check the plumbing first (offline, ~seconds, uses toy documents and a fake generator).
!python tests/smoke_test.py | tail -3

## 1. Clean and chunk documents

Look at what the cleaner does to *your* documents: PDFs often have headers, page numbers, hyphenation and tables that produce junk. If chunks look bad, fix the cleaning rules in `clean_text()` and document the issues in the report (the brief asks for data quality challenges).

In [ ]:
!python src/data_prep.py --raw_dir data/raw --out data/interim/chunks.jsonl

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
chunks = [json.loads(l) for l in open('data/interim/chunks.jsonl')]
df = pd.DataFrame(chunks)
print(df.groupby('crop').agg(chunks=('chunk_id', 'count'), docs=('source_doc', 'nunique'), words=('n_words', 'mean')).round(0))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df['crop'].value_counts().plot.bar(ax=ax[0], title='Chunks per crop')
df.explode('diseases')['diseases'].value_counts().head(15).plot.barh(ax=ax[1], title='Chunks per disease (top 15)')
plt.tight_layout(); plt.savefig('results/eda_chunks.png', dpi=150); plt.show()
for c in df.sample(3, random_state=0).itertuples():
    print('\n---', c.chunk_id, c.crop, c.diseases, '\n', c.text[:500])

## 2. Generate question-answer pairs (LLM-assisted)

Start with `--limit 16` to inspect quality before spending an hour on the full run. The script is **resumable**: if Colab disconnects, run the same command again.

Model choice: `Qwen/Qwen2.5-3B-Instruct` fits a T4 in fp16. A 7B model with `--load_in_4bit` gives better pairs but is slower. Report which model you used.

In [ ]:
!python src/build_qa.py --chunks data/interim/chunks.jsonl --out data/interim/qa_test_run.jsonl --limit 16 --pairs_per_chunk 3

In [ ]:
rows = [json.loads(l) for l in open('data/interim/qa_test_run.jsonl')]
for r in rows[:6]:
    print('Q:', r['question']); print('A:', r['answer']); print('   [', r['gold_chunk_id'], ']\n')

In [ ]:
# Full run (resumable). Delete data/interim/qa_test_run.* first if you do not want it to count.
!python src/build_qa.py --chunks data/interim/chunks.jsonl --out data/interim/qa_raw.jsonl --pairs_per_chunk 3

Read the **filter outcomes** line printed at the end: it shows how many generated pairs were rejected and why. Put this in your dataset section. If `ungrounded_number` or `low_overlap` dominate, the generator is hallucinating or paraphrasing too freely; try a larger model or a lower temperature.

In [ ]:
qa = pd.DataFrame([json.loads(l) for l in open('data/interim/qa_raw.jsonl')])
print(len(qa), 'pairs'); print(qa['crop'].value_counts()); print(qa['qtype'].value_counts())
qa['q_words'] = qa.question.str.split().str.len(); qa['a_words'] = qa.answer.str.split().str.len()
qa[['q_words', 'a_words']].hist(bins=20, figsize=(9, 3)); plt.tight_layout(); plt.savefig('results/eda_qa_lengths.png', dpi=150)

## 3. Manual verification (do not skip)

Automatic filters cannot tell whether an answer is *agronomically* correct. Export a sample, check each row against its passage, and report the error rate. For credibility, have a **second person** review ~30 of the same rows and report agreement.

1. Run the export cell, download `data/interim/review_sample.csv`, fill the `verdict` column (`correct`, `minor_issue`, `incorrect`, `unsupported`), optionally repair pairs in `fixed_question` / `fixed_answer`, and upload it back.
2. Run the apply cell.

In [ ]:
!python src/review.py export --qa data/interim/qa_raw.jsonl --n 150 --out data/interim/review_sample.csv

In [ ]:
# After reviewing the CSV:
!python src/review.py apply --qa data/interim/qa_raw.jsonl --reviewed data/interim/review_sample.csv --out data/interim/qa_clean.jsonl

## 4. Climate-conditioned examples (synthetic)

These come from the rules in `src/knowledge.py`. **Open that file and verify every threshold against the literature**, then cite your sources in the report. Vector-borne viruses are excluded on purpose.

In [ ]:
!python src/build_climate.py --chunks data/interim/chunks.jsonl --out data/interim/climate.jsonl

In [ ]:
cl = [json.loads(l) for l in open('data/interim/climate.jsonl')]
pair = [r for r in cl if r['pair_id']][:2]
for r in pair:
    print(r['climate'], '->', r['risk']); print(r['question']); print(r['answer'], '\n')

## 5. Out-of-domain examples

The seed lists are small. **Expand them** (more phrasing variety, local topics, real off-topic queries from classmates, maybe LLM paraphrases) and say how in the data card, otherwise the domain gate will look better on your test set than it will in real use.

In [ ]:
!python src/build_ood.py --out data/interim/ood.jsonl

## 6. Split by source document and attach retrieval context

This also prints a free **BM25 retrieval baseline (B1)**: Recall@k and MRR on val/test. If `bm25_recall@3` is low on the test split, answers will suffer regardless of the LLM, which is a finding for your error analysis.

In [ ]:
!python src/make_splits.py --chunks data/interim/chunks.jsonl --qa data/interim/qa_clean.jsonl --climate data/interim/climate.jsonl --ood data/interim/ood.jsonl --out_dir data/processed

## 7. Final checks

In [ ]:
for sp in ['train', 'val', 'test']:
    rows = [json.loads(l) for l in open(f'data/processed/{sp}.jsonl')]
    d = pd.DataFrame(rows)
    print(sp, len(d), dict(d['type'].value_counts()), '| crops:', dict(d[d.type != 'ood']['crop'].value_counts()))
print(open('data/processed/stats.json').read()[:1500])

In [ ]:
# Look at a few finished training rows exactly as the model will see them.
from prompting import build_messages
rows = [json.loads(l) for l in open('data/processed/train.jsonl')]
for r in [x for x in rows if x['type'] == 'climate'][:1] + [x for x in rows if x['type'] == 'ood'][:1]:
    print(build_messages(r)[1]['content']); print('-> TARGET:', r['answer'], '\n' + '=' * 60)

## What to write in the report's Dataset section
- Sources table (from `sources.csv`) and licences; why these documents are appropriate for Rwandan smallholders and where they are not.
- Chunking and cleaning decisions and the problems you hit (PDF extraction, tables, duplicate content).
- How QA pairs were generated (model, prompt, filters) with the **filter outcome counts** and the **manual review error rate**.
- That climate data are synthetic and rule-based, with the citations behind each rule.
- How the OOD set was built and expanded.
- Split-by-document strategy, the leakage checks, and the context policy (train gold+distractors vs test retrieved).
- Limitations (see `data/DATA_CARD.md`).